<a href="https://colab.research.google.com/github/Ilham-sy/psa-nlp-project/blob/main/NLP_GRP_PRJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PSA Translation Project

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd


# Define the base path for files in Google Drive
drive_path = "/content/drive/MyDrive/"

multilanguage = pd.read_csv(f"{drive_path}Multilanguage_PSA.csv")
PSA = pd.read_csv(f"{drive_path}PSA_KE_Final.csv")

In [ ]:
# Load the dataset
# This cell is redundant as data is loaded in StYVbsX22qZn
# multilanguage = pd.read_csv("Multilanguage_PSA.csv")
# PSA = pd.read_csv("PSA_KE_Final.csv")

In [ ]:
# Check the dimensions
print("Multilanguage:", multilanguage.shape)
print("PSA:", PSA.shape)

Multilanguage: (6648, 9)
PSA: (2903, 8)


In [ ]:
# Check the columns
print("Multilanguage columns:")
print(multilanguage.columns.tolist())

print("\nPSA columns:")
print(PSA.columns.tolist())

Multilanguage columns:
['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Source', 'Date', 'Metadata', 'PSA ID']

PSA columns:
['PSA_Id', 'Domain', 'Class', 'English', 'Kiswahili', 'Ekegusii', 'Dholuo', 'Somali']


In [ ]:
# Drop unnecessary columns for multilanguage
multilanguage = multilanguage.drop(columns=["PSA ID", "Source", "Date", "Metadata"])

In [ ]:
# Drop unnecessary columns for PSA
PSA = PSA.drop(columns=["Class", "Ekegusii", "Somali"])

In [ ]:
# Rename columns
PSA = PSA.rename(columns={"PSA_Id": "PSA_ID"})

In [ ]:
# Combine the datasets
combined = pd.concat([multilanguage, PSA], ignore_index=True)

In [ ]:
# check the combined dataset
print(combined.shape)

combined.head()

(9551, 5)


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN


In [ ]:
# Assign the combined DataFrame to 'df' for the cleaning process
df = combined

In [ ]:
combined.to_csv(
    f"{drive_path}Combined_PSA_Raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Structural Cleaning

In [ ]:
from google.colab import drive
import pandas as pd
import re

In [ ]:
# Merge duplicate ID columns (if both exist)
if 'PSA ID' in df.columns and 'PSA_ID' in df.columns:
    df['PSA_ID'] = df['PSA_ID'].fillna(df['PSA ID'])
    df = df.drop(columns=['PSA ID'])

TEXT_COLS = ['English', 'Kiswahili', 'Dholuo']

def clean_text(val):
    if pd.isna(val):
        return val

    text = str(val)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove HTML entities
    text = re.sub(r'&nbsp;|&amp;|&quot;|&#\d+;', ' ', text)

    # Clean special character encoding
    text = re.sub(r'_x0092_', "'", text)   # right single quote
    text = re.sub(r'_x0093_', '"', text)   # left double quote
    text = re.sub(r'_x0094_', '"', text)   # right double quote
    text = re.sub(r'_x0096_', '-', text)   # en dash
    text = re.sub(r'_x0097_', '-', text)   # em dash
    text = re.sub(r'_x00[0-9A-Fa-f]{2}_', ' ', text)  # catch-all

    # Remove line breaks and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # Convert curly quotes to straight quotes
    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")

    # Remove extra spaces around brackets
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Collapse multiple spaces
    text = re.sub(r'\s{2,}', ' ', text)

    # Trim whitespace
    text = text.strip()

    return text

# Apply cleaning
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# Remove rows with empty English text
if 'English' in df.columns:
    df = df[df['English'].notna() & (df['English'].str.strip() != '')]

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()  # exact full-row duplicates (safe to keep)

# Remove content-duplicate rows (same PSA under a different PSA_ID)
dedup_cols = ['English', 'Kiswahili'] # Removed 'Source' and 'Date'
df = df.drop_duplicates(subset=dedup_cols, keep='first')

print(f"Duplicate rows removed: {before - len(df)}")

# Save cleaned dataset back to Google Drive
output_path = '/content/drive/MyDrive/PSA_Clean_v1.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Final rows: {len(df)}")
print(f"Saved as: {output_path}")

# Preview cleaned data
df.head()

Duplicate rows removed: 9
Final rows: 9540
Saved as: /content/drive/MyDrive/PSA_Clean_v1.csv


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN


In [ ]:
df.shape

(9540, 5)

# Data Quality Cleaning

In [ ]:
import pandas as pd
import numpy as np
import re

input_path = "/content/drive/MyDrive/PSA_Clean_v1.csv"

df = pd.read_csv(input_path)

print("Rows before cleaning:", len(df))
print("Columns:", list(df.columns))

Rows before cleaning: 9540
Columns: ['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo']


In [ ]:
# Replace common missing value representations

missing_values = [
    "Unknown",
    "unknown",
    "UNKNOWN",
    "N/A",
    "n/a",
    "NA",
    "None",
    "null",
    ""
]

df = df.replace(missing_values, np.nan)

In [ ]:
# Clean text column
TEXT_COLS = ["English","Kiswahili","Dholuo"]

In [ ]:
def clean_text(text):

    if pd.isna(text):
        return text

    text = str(text)

    # remove emails
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)

    # remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # remove phone numbers
    text = re.sub(r'\+?\d[\d\-\s]{7,}\d', ' ', text)

    # remove website menus
    patterns = [

        r"Toggle navigation",
        r"Skip to main content",
        r"Home",
        r"Search website",
        r"Search",
        r"English",
        r"Français",
        r"Portuguese",
        r"About Us",
        r"About us",
        r"Contact us",
        r"Accessibility Statement",
        r"Privacy Policy",
        r"Cookie Policy",
        r"Terms of Use",
        r"Terms and Conditions",
        r"Copyright",

    ]

    for p in patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE)

    # remove multiple spaces

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
for col in TEXT_COLS:

    if col in df.columns:

        df[col] = df[col].apply(clean_text)

In [ ]:
# Remove empty English rows
df = df[
    df["English"].notna()
]

df = df[
    df["English"].str.strip() != ""
]

In [ ]:
# Remove duplicate rows
before = len(df)

df = df.drop_duplicates()

print("Duplicates removed:", before-len(df))

Duplicates removed: 0


In [ ]:
# Reset index
df = df.reset_index(drop=True)

In [ ]:
# Data summary
print("="*50)

print("Rows:",len(df))

print("\nMissing values")

print(df.isnull().sum())

print("="*50)

Rows: 9540

Missing values
PSA_ID        128
Domain          0
English         0
Kiswahili       3
Dholuo       6650
dtype: int64


In [ ]:
df[df["PSA_ID"].isna()].head(10)

,PSA_ID,Domain,English,Kiswahili,Dholuo
1483,NaN,Security,High traffic areas frequented by foreigners an...,Maeneo yenye msongamano mkubwa wa magari yanay...,NaN
1484,NaN,Security,Stay alert in locations frequented by tourists...,Kuwa macho katika maeneo yanayotembelewa na wa...,NaN
1485,NaN,Security,Review your personal security plans,Kagua mipango yako ya usalama wa kibinafsi,NaN
1486,NaN,Security,Be aware of your surroundings,Kuwa mwangalifu kuhusu mazingira yako,NaN
1487,NaN,Security,Monitor local media for updates,Fuatilia vyombo vya habari vya ndani kwa ajili...,NaN
1488,NaN,Security,Avoid protest areas and demonstrations,Epuka maeneo ya maandamano na maandamano,NaN
1489,NaN,Security,Avoid crowds,Epuka umati wa watu,NaN
1490,NaN,Security,Keep a low profile,Weka wasifu mdogo,NaN
1491,NaN,Security,Keep doors locked and windows rolled up while ...,Weka milango imefungwa na madirisha yamefungwa...,NaN
1492,NaN,Security,Notify friends and family of your whereabouts ...,Wajulishe marafiki na familia kuhusu mahali ul...,NaN


In [ ]:
output_path="/content/drive/MyDrive/PSA_Clean_v2.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")

print(output_path)

Saved successfully!
/content/drive/MyDrive/PSA_Clean_v2.csv


In [ ]:
print(df.head())

print(df.sample(10))

print(df.shape)

      PSA_ID  Domain                                            English  \
0  PSA000003  Health  PRESS RELEASE: JUNE 29, 2020 The EU through it...   
1  PSA000004  Health  This grant will be used by the WHO to support ...   
2  PSA000005  Health  Specifically, WHO Kenya will boost the respons...   
3  PSA000009  Health  Strengthening clinical care for high-consequen...   
4  PSA000010  Health  In recent years, investments in surveillance, ...   

                                           Kiswahili Dholuo  
0  TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...    NaN  
1  Ruzuku hii itatumiwa na WHO kuunga mkono juhud...    NaN  
2  Hasa, WHO Kenya itaongeza juhudi za kukabilian...    NaN  
3  Kuimarisha huduma ya kimatibabu kwa magonjwa y...    NaN  
4  Katika miaka ya hivi karibuni, uwekezaji katik...    NaN  
         PSA_ID       Domain  \
2585  PSA000975  Agriculture   
6872        283    Education   
9530       3766   Governance   
105   PSA000150       Health   
2855  PSA001245  

# PSA Content Validation

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/PSA_Clean_v2.csv")

print(df.shape)

(9540, 5)


In [ ]:
# Create a review column
df["Status"] = ""

In [ ]:
# Obvious non psa
remove_patterns = [
    "privacy policy",
    "cookie policy",
    "terms of use",
    "terms and conditions",
    "accessibility statement",
    "careers",
    "job vacancies",
    "contact us",
    "about us",
    "copyright",
    "all rights reserved",
    "toggle navigation",
    "search website",
    "skip to main content",
    "login",
    "register",
    "home page"
]

mask = df["English"].str.lower().str.contains(
    "|".join(remove_patterns),
    na=False
)

df.loc[mask, "Status"] = "Remove"

print(mask.sum(), "rows marked Remove")

514 rows marked Remove


In [ ]:
# Likely PSAs
psa_keywords = [

    "should",
    "must",
    "avoid",
    "protect",
    "prevent",
    "vaccinate",
    "wash",
    "seek",
    "report",
    "stay",
    "keep",
    "ensure",
    "call",
    "wear",
    "use",
    "remember",
    "monitor",
    "is advised",
    "are advised",
    "do not",
    "please",
    "do not",
    "must",
    "should",
    "ensure",
    "remember",
    "please",
    "seek medical",
    "wash your hands",
    "protect yourself",
    "keep away",
    "stay at home",
    "visit your nearest",
    "monitor",
    "be alert",
    "take precautions",
    "stay safe",
    "keep children",
    "use mosquito nets",
    "boil drinking water"

]

mask = df["English"].str.lower().str.contains(
    "|".join(psa_keywords),
    na=False
)

df.loc[
    (mask) &
    (df["Status"]==""),
    "Status"
] = "Possible PSA"

print(mask.sum(), "possible PSAs")

3707 possible PSAs


In [ ]:
df.loc[df["Status"]=="","Status"]="Needs Review"

In [ ]:
# Check the distribution
df["Status"].value_counts()

,count
Status,
Needs Review,5549
Possible PSA,3477
Remove,514


In [ ]:
# Possible review
possible = df[df["Status"]=="Possible PSA"]

possible.head(20)

,PSA_ID,Domain,English,Kiswahili,Dholuo,Status
0,PSA000003,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN,Possible PSA
1,PSA000004,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN,Possible PSA
2,PSA000005,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN,Possible PSA
8,PSA000014,Health,Caring for patients with high-consequence infe...,Kuwatunza wagonjwa wenye magonjwa ya kuambukiz...,NaN,Possible PSA
9,PSA000015,Health,These conditions are physically demanding and ...,Hali hizi zinahitajika kimwili na kitaalamu ni...,NaN,Possible PSA
11,PSA000017,Health,"To address this gap, countries are strengtheni...","Ili kushughulikia pengo hili, nchi zinaimarish...",NaN,Possible PSA
12,PSA000018,Health,"From 27 April to 1 May 2026, clinicians, infec...","Kuanzia tarehe 27 Aprili hadi 1 Mei 2026, mada...",NaN,Possible PSA
14,PSA000020,Health,"The programme focused on critical care, patien...","Programu hiyo ililenga katika utunzaji muhimu,...",NaN,Possible PSA
16,PSA000022,Health,World Health Organization supported this effor...,Shirika la Afya Duniani liliunga mkono juhudi ...,NaN,Possible PSA
20,PSA000026,Health,The next phase is to ensure that every patient...,Hatua inayofuata ni kuhakikisha kwamba kila mg...,NaN,Possible PSA


In [ ]:
# Needs review
review = df[df["Status"]=="Needs Review"]

review.head(20)

,PSA_ID,Domain,English,Kiswahili,Dholuo,Status
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN,Needs Review
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN,Needs Review
5,PSA000011,Health,"Across Tanzania, Uganda and Ethiopia, early wa...","Kote Tanzania, Uganda na Ethiopia, mifumo ya t...",NaN,Needs Review
6,PSA000012,Health,"Yet patient outcomes continue to vary, pointin...",Hata hivyo matokeo ya mgonjwa yanaendelea kuto...,NaN,Needs Review
7,PSA000013,Health,Clinical management has emerged as the determi...,Usimamizi wa kimatibabu umeibuka kama kigezo c...,NaN,Needs Review
10,PSA000016,Health,"Where bedside capacity is limited, mortality i...",Pale ambapo uwezo wa kando ya kitanda ni mdogo...,NaN,Needs Review
13,PSA000019,Health,"Organized by the East African Community, with ...",Mafunzo hayo yaliyoandaliwa na Jumuiya ya Afri...,NaN,Needs Review
15,PSA000021,Health,"Designed as a training-of-trainers platform, i...","Imeundwa kama jukwaa la mafunzo ya wakufunzi, ...",NaN,Needs Review
17,PSA000023,Health,This approach reflects a broader shift toward ...,Mbinu hii inaonyesha mabadiliko mapana kueleke...,NaN,Needs Review
18,PSA000024,Health,Rather than rebuilding capacity during each em...,Badala ya kujenga upya uwezo wakati wa kila dh...,NaN,Needs Review


In [ ]:
output_path = "/content/drive/MyDrive/PSA_Clean_v3.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")
print(output_path)

Saved successfully!
/content/drive/MyDrive/PSA_Clean_v3.csv


In [ ]:
# Filter only the rows that need manual review
needs_review = df[df["Status"] == "Needs Review"]

print("Rows to review:", len(needs_review))

Rows to review: 5549


In [ ]:
needs_review.head()

,PSA_ID,Domain,English,Kiswahili,Dholuo,Status
3,PSA000009,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,NaN,Needs Review
4,PSA000010,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...",NaN,Needs Review
5,PSA000011,Health,"Across Tanzania, Uganda and Ethiopia, early wa...","Kote Tanzania, Uganda na Ethiopia, mifumo ya t...",NaN,Needs Review
6,PSA000012,Health,"Yet patient outcomes continue to vary, pointin...",Hata hivyo matokeo ya mgonjwa yanaendelea kuto...,NaN,Needs Review
7,PSA000013,Health,Clinical management has emerged as the determi...,Usimamizi wa kimatibabu umeibuka kama kigezo c...,NaN,Needs Review


In [ ]:
output_path = "/content/drive/MyDrive/Needs_Review.csv"

needs_review.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/Needs_Review.csv


In [ ]:
# Split the dataset into 4 equal datasets
import numpy as np

# Split the dataframe into 4 approximately equal parts
splits = np.array_split(df, 4)

# Save each split
for i, split in enumerate(splits, start=1):
    filename = f"/content/drive/MyDrive/Needs_Review_Part_{i}.csv"
    split.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved: {filename} ({len(split)} rows)")

Saved: /content/drive/MyDrive/Needs_Review_Part_1.csv (2385 rows)
Saved: /content/drive/MyDrive/Needs_Review_Part_2.csv (2385 rows)
Saved: /content/drive/MyDrive/Needs_Review_Part_3.csv (2385 rows)
Saved: /content/drive/MyDrive/Needs_Review_Part_4.csv (2385 rows)


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
# Combining the reviewed datasets
import pandas as pd

# Load the four reviewed files
r1 = pd.read_csv("Reviewed 1.csv")
r2 = pd.read_csv("Reviewed 2.csv")
r3 = pd.read_csv("Reviewed 3.csv")
r4 = pd.read_csv("Reviewed 4.csv")

# Combine them
Reviewed = pd.concat([r1, r2, r3, r4], ignore_index=True)

# Remove completely empty columns (Unnamed columns)
Reviewed = Reviewed.loc[:, ~Reviewed.columns.str.contains("^Unnamed")]

# Save the combined dataset
Reviewed.to_csv("/content/drive/MyDrive/Reviewed.csv", index=False)

print("Rows:", Reviewed.shape[0])
print("Columns:", Reviewed.shape[1])

Rows: 4920
Columns: 8


In [ ]:
import pandas as pd

# Load datasets
v3 = pd.read_csv("/content/drive/MyDrive/PSA_Clean_v3.csv")
reviewed = pd.read_csv("/content/drive/MyDrive/Reviewed.csv")

In [ ]:
print(v3.columns.tolist())
print(reviewed.columns.tolist())

['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Status']
['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Status', 'Final_Status', 'Final_status']


In [ ]:
# Combine the two Final_Status columns
reviewed["Final_Status"] = reviewed["Final_Status"].fillna(reviewed["Final_status"])

# Drop the duplicate column
reviewed.drop(columns=["Final_status"], inplace=True)

In [ ]:
print(reviewed.columns.tolist())
print(reviewed["Final_Status"].value_counts(dropna=False))

['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Status', 'Final_Status']
Final_Status
Keep      2515
Remove    2405
Name: count, dtype: int64


In [ ]:
# Convert Status to Final_Status in v3
v3["Final_Status"] = v3["Status"].replace({
    "Possible PSA": "Keep",
    "Remove": "Remove"
})

In [ ]:
v3["Final_Status"].value_counts(dropna=False)

,count
Final_Status,
Needs Review,5549
Keep,3477
Remove,514


In [ ]:
# Remove Needs Review from v3
v3 = v3[v3["Final_Status"] != "Needs Review"].copy()

In [ ]:
v3["Final_Status"].value_counts()

,count
Final_Status,
Keep,3477
Remove,514


In [ ]:
v3.shape

(3991, 7)

In [ ]:
# Keep only these columns
v3 = v3[[
    "PSA_ID",
    "Domain",
    "English",
    "Kiswahili",
    "Dholuo",
    "Final_Status"
]]

reviewed = reviewed[[
    "PSA_ID",
    "Domain",
    "English",
    "Kiswahili",
    "Dholuo",
    "Final_Status"
]]

In [ ]:
print(v3.columns.tolist())
print(reviewed.columns.tolist())

['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Final_Status']
['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Final_Status']


In [ ]:
# Combine the two datasets
final_dataset = pd.concat([v3, reviewed], ignore_index=True)

In [ ]:
print(final_dataset.shape)
print(final_dataset["Final_Status"].value_counts())

(8911, 6)
Final_Status
Keep      5992
Remove    2919
Name: count, dtype: int64


In [ ]:
# Keep only PSAs
final_dataset = final_dataset[
    final_dataset["Final_Status"] == "Keep"
].copy()

In [ ]:
print(final_dataset.shape)
print(final_dataset["Final_Status"].value_counts())

(5992, 6)
Final_Status
Keep    5992
Name: count, dtype: int64


In [ ]:
# Renumber PSA ID
final_dataset = final_dataset.reset_index(drop=True)

final_dataset["PSA_ID"] = [
    f"PSA{str(i+1).zfill(6)}"
    for i in range(len(final_dataset))
]

In [ ]:
final_dataset.head()

,PSA_ID,Domain,English,Kiswahili,Dholuo,Final_Status
0,PSA000001,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...",NaN,Keep
1,PSA000002,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,NaN,Keep
2,PSA000003,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...",NaN,Keep
3,PSA000004,Health,Caring for patients with high-consequence infe...,Kuwatunza wagonjwa wenye magonjwa ya kuambukiz...,NaN,Keep
4,PSA000005,Health,These conditions are physically demanding and ...,Hali hizi zinahitajika kimwili na kitaalamu ni...,NaN,Keep


In [ ]:
# Save the final dataset
final_dataset.to_csv(
    "/content/drive/MyDrive/PSA_Final.csv",
    index=False
)